# Pré-processamento e Modelagem — Risco de Crédito

Este notebook aplica o pré-processamento definido a partir da EDA e treina os cinco modelos de classificação (KNN, Árvore de Decisão, Random Forest, AdaBoost e MLP), comparando os resultados com o benchmark de Yang et al. (2025).


## 1. Importação e carregamento dos dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Carregar a tabela principal
df = pd.read_csv('../data/application_train.csv')

print(f"Dados carregados: {df.shape[0]} linhas e {df.shape[1]} colunas")

Dados carregados: 307511 linhas e 122 colunas


## 2. Tratamento da anomalia em DAYS_EMPLOYED

O valor 365.243 dias (~1000 anos) é um código placeholder presente em ~18% dos registros, indicando clientes sem vínculo empregatício. Substituímos por NaN e criamos uma flag binária, preservando essa informação como preditor.

In [2]:
# Criar flag que marca os registros com a anomalia (antes de substituir)
df['FLAG_SEM_EMPREGO'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)

# Substituir o valor anômalo por NaN
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# Verificar o resultado
print("Flag criada - distribuição:")
print(df['FLAG_SEM_EMPREGO'].value_counts())
print(f"\nValores faltantes em DAYS_EMPLOYED agora: {df['DAYS_EMPLOYED'].isnull().sum()}")
print(f"Novo máximo de DAYS_EMPLOYED: {df['DAYS_EMPLOYED'].max():.0f} dias")

Flag criada - distribuição:
FLAG_SEM_EMPREGO
0    252137
1     55374
Name: count, dtype: int64

Valores faltantes em DAYS_EMPLOYED agora: 55374
Novo máximo de DAYS_EMPLOYED: 0 dias


## 3. Tratamento de valores faltantes

Estratégia definida a partir da EDA: descartar colunas com proporção excessiva de faltantes (> 60%) e imputar as demais: Mediana para variáveis numéricas (robusta a outliers) e moda para categóricas.

In [3]:
# Calcular o percentual de faltantes por coluna
pct_faltantes = (df.isnull().sum() / len(df)) * 100

# Identificar colunas com mais de 60% de faltantes
colunas_descartar = pct_faltantes[pct_faltantes > 60].index.tolist()

print(f"Colunas com mais de 60% de faltantes (serão descartadas): {len(colunas_descartar)}")
print(colunas_descartar)

df = df.drop(columns=colunas_descartar)

print(f"\nFormato após descarte: {df.shape[0]} linhas e {df.shape[1]} colunas")

Colunas com mais de 60% de faltantes (serão descartadas): 17
['OWN_CAR_AGE', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'FLOORSMIN_AVG', 'LIVINGAPARTMENTS_AVG', 'NONLIVINGAPARTMENTS_AVG', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'FLOORSMIN_MODE', 'LIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_MODE', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'FLOORSMIN_MEDI', 'LIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'FONDKAPREMONT_MODE']

Formato após descarte: 307511 linhas e 106 colunas


### 3.1 Separação de colunas numéricas e categóricas

As colunas são separadas por tipo, pois a imputação difere: numéricas recebem a mediana; categóricas, a moda.

In [4]:
# Separar colunas por tipo
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = df.select_dtypes(include=['object']).columns.tolist()

print(f"Colunas numéricas: {len(colunas_numericas)}")
print(f"Colunas categóricas: {len(colunas_categoricas)}")
print(f"\nExemplos de categóricas: {colunas_categoricas[:5]}")

Colunas numéricas: 91
Colunas categóricas: 15

Exemplos de categóricas: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE']


### 3.2 Imputação dos valores faltantes

Numéricas: preenchidas com a mediana (robusta a outliers). 

Categóricas: preenchidas com a moda (valor mais frequente).

In [5]:
# Imputar colunas NUMÉRICAS com a mediana
for col in colunas_numericas:
    if df[col].isnull().sum() > 0:
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)

# Imputar colunas CATEGÓRICAS com a moda (valor mais frequente)
for col in colunas_categoricas:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()[0]
        df[col] = df[col].fillna(moda)

# Verificar se ainda restam faltantes
total_faltantes = df.isnull().sum().sum()
print(f"Total de valores faltantes restantes: {total_faltantes}")

Total de valores faltantes restantes: 0


## 4. Codificação das variáveis categóricas

As variáveis de texto são convertidas em números. Estratégia: Label Encoding para binárias e One-Hot Encoding para as demais (categorias sem ordem natural).

In [6]:
# Investigar quantas categorias únicas cada coluna categórica tem
print("Número de categorias únicas por coluna categórica:\n")
for col in colunas_categoricas:
    n_unicas = df[col].nunique()
    print(f"{col}: {n_unicas} categorias")

Número de categorias únicas por coluna categórica:

NAME_CONTRACT_TYPE: 2 categorias
CODE_GENDER: 3 categorias
FLAG_OWN_CAR: 2 categorias
FLAG_OWN_REALTY: 2 categorias
NAME_TYPE_SUITE: 7 categorias
NAME_INCOME_TYPE: 8 categorias
NAME_EDUCATION_TYPE: 5 categorias
NAME_FAMILY_STATUS: 6 categorias
NAME_HOUSING_TYPE: 6 categorias
OCCUPATION_TYPE: 18 categorias
WEEKDAY_APPR_PROCESS_START: 7 categorias
ORGANIZATION_TYPE: 58 categorias
HOUSETYPE_MODE: 3 categorias
WALLSMATERIAL_MODE: 7 categorias
EMERGENCYSTATE_MODE: 2 categorias


In [7]:
# Investigar os valores da coluna CODE_GENDER
print("Valores em CODE_GENDER:")
print(df['CODE_GENDER'].value_counts())

Valores em CODE_GENDER:
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64


### 4.1 Limpeza do CODE_GENDER

A categoria "XNA" (não informado, apenas 4 registros) é substituída pela moda ("F"), tornando a variável binária.

In [8]:
# Substituir XNA (apenas 4 registros) pela moda
df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', 'F')

# Confirmar que agora só há 2 categorias
print("CODE_GENDER após limpeza:")
print(df['CODE_GENDER'].value_counts())

CODE_GENDER após limpeza:
CODE_GENDER
F    202452
M    105059
Name: count, dtype: int64


### 4.2 Label Encoding das variáveis binárias

As variáveis com apenas 2 categorias são convertidas para 0 e 1.

In [9]:
from sklearn.preprocessing import LabelEncoder

# Colunas binárias (2 categorias cada)
colunas_binarias = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
                    'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE']

# Aplicar Label Encoding em cada uma
le = LabelEncoder()
for col in colunas_binarias:
    df[col] = le.fit_transform(df[col])
    print(f"{col}: {df[col].unique()}")

print("\nLabel Encoding aplicado nas binárias!")

NAME_CONTRACT_TYPE: [0 1]
CODE_GENDER: [1 0]
FLAG_OWN_CAR: [0 1]
FLAG_OWN_REALTY: [1 0]
EMERGENCYSTATE_MODE: [0 1]

Label Encoding aplicado nas binárias!


### 4.3 Label Encoding das variáveis com muitas categorias

`OCCUPATION_TYPE` (18) e `ORGANIZATION_TYPE` (58) recebem Label Encoding para evitar a explosão de colunas do One-Hot. Justifica-se pelo uso predominante de modelos baseados em árvores, robustos a esse tipo de codificação.

In [10]:
# Label Encoding nas colunas com muitas categorias
colunas_muitas_cat = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

for col in colunas_muitas_cat:
    df[col] = le.fit_transform(df[col])
    print(f"{col}: convertida para valores de 0 a {df[col].max()}")

print("\nLabel Encoding aplicado nas colunas grandes!")

OCCUPATION_TYPE: convertida para valores de 0 a 17
ORGANIZATION_TYPE: convertida para valores de 0 a 57

Label Encoding aplicado nas colunas grandes!


### 4.4 One-Hot Encoding das demais categóricas

As variáveis categóricas sem ordem natural e com poucas categorias são convertidas via One-Hot Encoding, criando uma coluna binária para cada categoria.

In [11]:
# Colunas que receberão One-Hot Encoding
colunas_onehot = ['NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
                  'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
                  'WEEKDAY_APPR_PROCESS_START', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']

# Guardar o número de colunas antes
colunas_antes = df.shape[1]

# Aplicar One-Hot Encoding
df = pd.get_dummies(df, columns=colunas_onehot, drop_first=True)

# Converter colunas booleanas (True/False) para inteiros (1/0)
df = df.astype({col: 'int' for col in df.select_dtypes(include='bool').columns})

print(f"Colunas antes do One-Hot: {colunas_antes}")
print(f"Colunas depois do One-Hot: {df.shape[1]}")
print(f"Novas colunas criadas: {df.shape[1] - colunas_antes}")

Colunas antes do One-Hot: 106
Colunas depois do One-Hot: 139
Novas colunas criadas: 33


## 5. Separação de features (X) e alvo (y)

Os dados são divididos entre as variáveis preditoras (X) e a variável-alvo (y). O identificador `SK_ID_CURR` é removido por não ter valor preditivo.

In [12]:
# Separar features (X) e alvo (y)
# Remover TARGET (alvo) e SK_ID_CURR (apenas identificador)
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f"X (features): {X.shape[0]} linhas e {X.shape[1]} colunas")
print(f"y (alvo): {y.shape[0]} valores")
print(f"\nDistribuição do alvo:")
print(y.value_counts(normalize=True) * 100)

X (features): 307511 linhas e 137 colunas
y (alvo): 307511 valores

Distribuição do alvo:
TARGET
0    91.927118
1     8.072882
Name: proportion, dtype: float64


## 6. Divisão treino/teste (Holdout)

Os dados são divididos em 80% para treino e 20% para teste. Usa-se estratificação (`stratify`) para preservar a proporção de classes em ambos os conjuntos, e uma semente fixa (`random_state`) para reprodutibilidade.

In [13]:
from sklearn.model_selection import train_test_split

# Dividir em treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,    # semente fixa: garante que a divisão seja sempre a mesma
    stratify=y          # mantém a proporção de classes (8% inadimplentes) nos dois
)

print(f"Treino: {X_train.shape[0]} linhas")
print(f"Teste:  {X_test.shape[0]} linhas")
print(f"\nProporção de inadimplentes no treino: {y_train.mean()*100:.2f}%")
print(f"Proporção de inadimplentes no teste:  {y_test.mean()*100:.2f}%")

Treino: 246008 linhas
Teste:  61503 linhas

Proporção de inadimplentes no treino: 8.07%
Proporção de inadimplentes no teste:  8.07%


## 7. Padronização das features

As features são padronizadas (média 0, desvio 1) com `StandardScaler`, essencial para modelos sensíveis a escala (KNN, MLP). O scaler é ajustado apenas no treino e aplicado a treino e teste, evitando vazamento de dados.

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Ajustar (fit) APENAS no treino e transformar o treino
X_train_scaled = scaler.fit_transform(X_train)

# Apenas transformar o teste (usando os parâmetros aprendidos no treino)
X_test_scaled = scaler.transform(X_test)

print("Padronização concluída!")
print(f"Média das features no treino (deve ser ~0): {X_train_scaled.mean():.4f}")
print(f"Desvio padrão no treino (deve ser ~1): {X_train_scaled.std():.4f}")

Padronização concluída!
Média das features no treino (deve ser ~0): 0.0000
Desvio padrão no treino (deve ser ~1): 1.0000


## 8. Balanceamento com SMOTE

Para corrigir o desbalanceamento (8% de inadimplentes), aplica-se o SMOTE, que gera exemplos sintéticos da classe minoritária. **Aplicado apenas ao treino**, preservando a distribuição real no teste para uma avaliação fidedigna.

In [15]:
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f"Antes do SMOTE - distribuição no treino: {Counter(y_train)}")

# Aplicar SMOTE APENAS no treino
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Depois do SMOTE - distribuição no treino: {Counter(y_train_balanced)}")
print(f"\nTotal de amostras no treino após SMOTE: {len(y_train_balanced)}")

Antes do SMOTE - distribuição no treino: Counter({0: 226148, 1: 19860})
Depois do SMOTE - distribuição no treino: Counter({0: 226148, 1: 226148})

Total de amostras no treino após SMOTE: 452296
